# Algorithm Validation

**Goal:** compare algorithms using a test set to test the accuracy of our models

# Note for ELLY 
**Kernel Crash** The kernel keeps crashing and I was trying to figure out why it is not working and I am struggling with that. There is probably something with a high time complexity that is taking time but I am not sure what this function is. There may also be a function that is trying to read the whole file?? maybe? I am not sure

In [2]:
## 1) Load a fresh validation set (~40k rows not used for training)

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

VALIDATION_ROWS = 20000
RAW_PATH = "../Data/raw/merged_trip_emissions_coordinates.csv"
CHUNK_SIZE = 100000

temp_chunks = []
rows_accum = 0
for chunk in pd.read_csv(RAW_PATH, chunksize=CHUNK_SIZE, low_memory=False):
    temp_chunks.append(chunk)
    rows_accum += len(chunk)
    if rows_accum >= VALIDATION_ROWS * 2:
        break

val_df = pd.concat(temp_chunks, ignore_index=True)

if len(val_df) < VALIDATION_ROWS:
    raise ValueError(f"Not enough rows in validation file: {len(val_df)} found, {VALIDATION_ROWS} required")

val_df = val_df.sample(n=VALIDATION_ROWS, random_state=42).reset_index(drop=True)

## Feature engineering (same features used in your clustering pipelines)
val_df["pickup_datetime"] = pd.to_datetime(val_df["pickup_datetime"], errors="coerce")
val_df["pickup_hour"] = val_df["pickup_datetime"].dt.hour.fillna(0).astype(int)
val_df["hour_sin"] = np.sin(2 * np.pi * val_df["pickup_hour"] / 24)
val_df["hour_cos"] = np.cos(2 * np.pi * val_df["pickup_hour"] / 24)

FEATURE_COLS = ["pickup_lat", "pickup_lon", "hour_sin", "hour_cos"]
val_df = val_df.dropna(subset=FEATURE_COLS).reset_index(drop=True)

X_val = val_df[FEATURE_COLS].astype(float)
scaler = StandardScaler()
X_val_scaled = scaler.fit_transform(X_val)

print(f"Validation data: {len(val_df)} rows, features: {FEATURE_COLS}")

Validation data: 20000 rows, features: ['pickup_lat', 'pickup_lon', 'hour_sin', 'hour_cos']


In [3]:
# 2) Apply clustering models to the validation set (memory-safe workflow)
# Reduce effective sample size for heavier algorithms when kernel memory is limited.
RUN_ROWS = min(10000, len(X_val_scaled))  # manage O(n^2) cost for hierarchical and DBSCAN
X_small = X_val_scaled[:RUN_ROWS]

KMEANS_CLUSTERS = 29
HIER_CLUSTERS = 29
DBSCAN_EPS = 0.333
DBSCAN_MIN_SAMPLES = 8

# KMeans is relatively scalable; run on full subsample.
from sklearn.cluster import MiniBatchKMeans

kmeans = MiniBatchKMeans(n_clusters=KMEANS_CLUSTERS, random_state=42, batch_size=2048, n_init=10)
km_labels = kmeans.fit_predict(X_small)

# Hierarchical is expensive; run on a smaller subset to avoid O(N^2) explosion.
hier = AgglomerativeClustering(n_clusters=HIER_CLUSTERS, linkage="ward")
hier_labels = hier.fit_predict(X_small)

# DBSCAN can also be heavy for dense 40k data; run on same smaller subsample.
dbscan = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES, metric="euclidean", n_jobs=-1)
db_labels = dbscan.fit_predict(X_small)

validation_counts = {
    "kmeans": len(set(km_labels)),
    "hierarchical": len(set(hier_labels)),
    "dbscan": len(set(db_labels) - {-1}),
    "dbscan_noise_pct": float((db_labels == -1).mean() * 100),
}

print(f"Using {RUN_ROWS} rows for metric calculations.")
print("Cluster counts:", validation_counts)

# 3) Compute silhouette score and Davies-Bouldin index for each algorithm (subset metrics)


def compute_metrics(X, labels, label_name):
    stats = {
        "algorithm": label_name,
        "n_clusters": int(len(set(labels) - {-1})) if -1 in set(labels) else int(len(set(labels))),
        "silhouette": None,
        "davies_bouldin": None,
        "noise_pct": 0.0,
    }

    if label_name == "DBSCAN":
        outliers = labels == -1
        stats["noise_pct"] = float(np.mean(outliers) * 100)
        mask = ~outliers
    else:
        mask = np.ones(len(labels), dtype=bool)

    if mask.sum() < 2 or len(np.unique(labels[mask])) < 2:
        return stats

    stats["silhouette"] = float(silhouette_score(X[mask], labels[mask], sample_size=min(10000, mask.sum()), random_state=42))
    stats["davies_bouldin"] = float(davies_bouldin_score(X[mask], labels[mask]))
    return stats

results = [
    compute_metrics(X_small, km_labels, "KMeans"),
    compute_metrics(X_small, hier_labels, "Hierarchical"),
    compute_metrics(X_small, db_labels, "DBSCAN"),
]

results_df = pd.DataFrame(results)
results_df["silhouette"] = results_df["silhouette"].round(4)
results_df["davies_bouldin"] = results_df["davies_bouldin"].round(4)
results_df["noise_pct"] = results_df["noise_pct"].round(2)

print(f"\nValidation metrics (computed on first {RUN_ROWS} rows subset):")
print(results_df.to_string(index=False))

# Save results for later review
results_df.to_csv("../Data/generated/validation_clustering_metrics.csv", index=False)
print("\nSaved: ../Data/generated/validation_clustering_metrics.csv")

Using 10000 rows for metric calculations.
Cluster counts: {'kmeans': 29, 'hierarchical': 29, 'dbscan': 112, 'dbscan_noise_pct': 13.819999999999999}

Validation metrics (computed on first 10000 rows subset):
   algorithm  n_clusters  silhouette  davies_bouldin  noise_pct
      KMeans          29      0.2489          1.0823       0.00
Hierarchical          29      0.2051          1.1591       0.00
      DBSCAN         112     -0.0631          1.7312      13.82

Saved: ../Data/generated/validation_clustering_metrics.csv
